In [ ]:
import numpy as np
import pandas as pd

from functools import partial
import importlib

import src.data
import src.features
import src.kernel_normalisation
import src.kernels
import src.krr
import src.metrics
import src.multikernel

from src.data import encode_labels, load_data, train_val_split
from src.features import HOG, HSVHistogram, ColorMoments
from src.kernel_normalisation import unit_diag
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.metrics import accuracy
from src.multikernel import *

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### HOG features

In [5]:
hog = HOG(
    image_shape=(3, 32, 32),
    colour_mode="rgb",
    cell_size=8,
    block_size=2,
    block_stride=1,
    num_bins=6,
    signed=False,
).fit(X_tr)

X_tr_hog = hog.transform(X_tr)
X_te_hog = hog.transform(X_te)

X_train_hog, X_val_hog, y_train_hog, y_val_hog = train_val_split(
    X_tr_hog, y_tr, test_size=0.3, seed=20
)

### HSV Histogram features

In [14]:
hsv = HSVHistogram(bins=(8, 8, 8), grid=(4, 4)).fit(X_tr)

X_tr_hsv = hsv.transform(X_tr)
X_te_hsv = hsv.transform(X_te)

X_train_hsv, X_val_hsv, y_train_hsv, y_val_hsv = train_val_split(
    X_tr_hsv, y_tr, test_size=0.3, seed=20
)

### Color moment features

In [4]:
cm = ColorMoments(grid=(8, 8)).fit(X_tr)

X_tr_cm = cm.transform(X_tr)
X_te_cm = cm.transform(X_te)

X_train_cm, X_val_cm, y_train_cm, y_val_cm = train_val_split(
    X_tr_cm, y_tr, test_size=0.3, seed=20
)

Grid search over kernel weights

In [15]:
gammas_hog = estimate_chi2_gammas_channel(X_train_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

gamma_hsv = estimate_gamma(X_train_hsv)
hsv_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_hsv)

gamma_cm = estimate_laplacian_gamma(X_train_cm)
cm_laplacian = partial(laplacian_kernel_matrix, gamma=gamma_cm)

specs3: list[KernelSpec] = [
    {
        "name": "hog_chi2",
        "Z": X_train_hog,
        "kernel_fn": hog_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "hsv_gaussian",
        "Z": X_train_hsv,
        "kernel_fn": hsv_gaussian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "cm_laplacian",
        "Z": X_train_cm,
        "kernel_fn": cm_laplacian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
]

# Local grid around ~ (hog≈0.6, cm≈0.2, hsv≈0.2)
# We grid (w_hog, w_cm); w_hsv = 1 - w_hog - w_cm.
w_hog_vals = np.array([0.5, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80], dtype=np.float32)
w_cm_vals = np.array([0.05, 0.10, 0.15, 0.20, 0.25], dtype=np.float32)

out3 = weight_grid_search_cv_three_kernels(
    specs3,
    y_train_hog,
    n_classes=n_classes,
    model="krr",
    w12_grid=(w_hog_vals.tolist(), w_cm_vals.tolist()),
    k=5,
    seed=20,
    lam=1e-4,
    verbose=1,
)

print("best_w (hog, hsv, cm):", out3["best_w"], "best_mean_acc:", out3["best_mean_acc"])

w=(0.500,0.050,0.450) mean_acc=0.5963
w=(0.500,0.100,0.400) mean_acc=0.5991
w=(0.500,0.150,0.350) mean_acc=0.6037
w=(0.500,0.200,0.300) mean_acc=0.6040
w=(0.500,0.250,0.250) mean_acc=0.6069
w=(0.550,0.050,0.400) mean_acc=0.5966
w=(0.550,0.100,0.350) mean_acc=0.6026
w=(0.550,0.150,0.300) mean_acc=0.6046
w=(0.550,0.200,0.250) mean_acc=0.6080
w=(0.550,0.250,0.200) mean_acc=0.6071
w=(0.600,0.050,0.350) mean_acc=0.5966
w=(0.600,0.100,0.300) mean_acc=0.6003
w=(0.600,0.150,0.250) mean_acc=0.6043
w=(0.600,0.200,0.200) mean_acc=0.6051
w=(0.600,0.250,0.150) mean_acc=0.6020
w=(0.650,0.050,0.300) mean_acc=0.5983
w=(0.650,0.100,0.250) mean_acc=0.6003
w=(0.650,0.150,0.200) mean_acc=0.6034
w=(0.650,0.200,0.150) mean_acc=0.6037
w=(0.650,0.250,0.100) mean_acc=0.6034
w=(0.700,0.050,0.250) mean_acc=0.5971
w=(0.700,0.100,0.200) mean_acc=0.6014
w=(0.700,0.150,0.150) mean_acc=0.6029
w=(0.700,0.200,0.100) mean_acc=0.6023
w=(0.700,0.250,0.050) mean_acc=0.6023
w=(0.750,0.050,0.200) mean_acc=0.5943
w=(0.750,0.1

## After tuning parameters, train the best model.

### Check the best model

In [21]:
w_hog, w_hsv, w_cm = np.asarray(out3["best_w"]).tolist()

# IMPORTANT: use the partial kernels you defined (with gammas)
Kh_tr = hog_chi2(X_train_hog, X_train_hog)
Ks_tr = hsv_gaussian(X_train_hsv, X_train_hsv)
Kc_tr = cm_laplacian(X_train_cm, X_train_cm)

# Match CV: unit-diagonal normalisation per kernel
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_train_hog))
Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_train_hsv))
Kc_tr = normalise_train_gram(Kc_tr, unit_diag(X_train_cm))

Ktr = w_hog * Kh_tr + w_hsv * Ks_tr + w_cm * Kc_tr

# VAL x TRAIN cross Grams
Kh_val = hog_chi2(X_val_hog, X_train_hog)
Ks_val = hsv_gaussian(X_val_hsv, X_train_hsv)
Kc_val = cm_laplacian(X_val_cm, X_train_cm)

Kh_val = normalise_cross_gram(Kh_val, unit_diag(X_val_hog), unit_diag(X_train_hog))
Ks_val = normalise_cross_gram(Ks_val, unit_diag(X_val_hsv), unit_diag(X_train_hsv))
Kc_val = normalise_cross_gram(Kc_val, unit_diag(X_val_cm), unit_diag(X_train_cm))

K_val = w_hog * Kh_val + w_hsv * Ks_val + w_cm * Kc_val

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_train_hog, K=Ktr
)
yhat_val, _ = best_model.predict(K_star=K_val)
print("val acc:", accuracy(y_val_hog, yhat_val))

val acc: 0.622


### Train on full dataset for submission

In [22]:
w_hog, w_hsv, w_cm = np.asarray(out3["best_w"]).tolist()

# Re-estimate gammas on FULL training set
gammas_hog = estimate_chi2_gammas_channel(X_tr_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

gamma_hsv = estimate_chi2_gamma(X_tr_hsv)
hsv_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_hsv)

gamma_cm = estimate_laplacian_gamma(X_tr_cm)
cm_laplacian = partial(laplacian_kernel_matrix, gamma=gamma_cm)


# FULL train Grams
Kh_tr = hog_chi2(X_tr_hog, X_tr_hog)
Ks_tr = hsv_gaussian(X_tr_hsv, X_tr_hsv)
Kc_tr = cm_laplacian(X_tr_cm, X_tr_cm)

Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_tr_hog))
Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_tr_hsv))
Kc_tr = normalise_train_gram(Kc_tr, unit_diag(X_tr_cm))

Ktr = w_hog * Kh_tr + w_cm * Kc_tr + w_hsv * Ks_tr

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_tr, K=Ktr
)

# TEST x TRAIN cross Grams
Kh_te_tr = hog_chi2(X_te_hog, X_tr_hog)
Ks_te_tr = hsv_gaussian(X_te_hsv, X_tr_hsv)
Kc_te_tr = cm_laplacian(X_te_cm, X_tr_cm)

Kh_te_tr = normalise_cross_gram(Kh_te_tr, unit_diag(X_te_hog), unit_diag(X_tr_hog))
Kc_te_tr = normalise_cross_gram(Kc_te_tr, unit_diag(X_te_cm), unit_diag(X_tr_cm))
Ks_te_tr = normalise_cross_gram(Ks_te_tr, unit_diag(X_te_hsv), unit_diag(X_tr_hsv))

K_star = w_hog * Kh_te_tr + w_cm * Kc_te_tr + w_hsv * Ks_te_tr

yte_int, _ = best_model.predict(K_star=K_star)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")